In [2]:
#!/usr/bin/env Rscript
##
## STEP2_variable_clustering.R
## ---------------------------
## - Variable-wise clustering on MDS space (isoMDS) for A–D datasets
## - Uses NbClust + Ward.D2 with Euclidean distance
## - Outputs:
##   * nbclust_var_labels_<ID>_YYYYMMDD.csv
##   * nbclust_var_summary_<ID>_YYYYMMDD.csv
##   * hclust_var_wardD2_<ID>_YYYYMMDD.rds
##

suppressPackageStartupMessages({
  library(NbClust)
})

## ---------------------------------------------------------------
## 0. Global settings
## ---------------------------------------------------------------

set.seed(42)  # 固定乱数シード

## ROOT パス（必要に応じて書き換えてください）
ROOT <- "~/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"

## 入出力ディレクトリ
INPUT_BASE  <- file.path(ROOT, "data", "calc_core_var", "01_distance_mds_variables")
OUTPUT_BASE <- file.path(ROOT, "data", "calc_core_var", "02_clustering_variables")

## MDS 次元数（例：10次元）
mds_dim_for_clustering <- 10

## NbClust 設定
min_nc <- 2
max_nc <- 25
nb_indices <- c("silhouette", "dunn", "gap", "ch", "db", "ptbiserial")

## データセット定義
datasets <- list(
  list(id = "A", subdir = "A_OH_plus_others"),
  list(id = "B", subdir = "B_FP_plus_others"),
  list(id = "C", subdir = "C_OH_only"),
  list(id = "D", subdir = "D_FP_only")
)

## 日付スタンプ
run_date <- format(Sys.Date(), "%Y%m%d")


## ---------------------------------------------------------------
## 1. Helper functions
## ---------------------------------------------------------------

# 指定パターンにマッチする「最新」のファイルを取得
find_latest_file <- function(dir_path, pattern) {
  files <- list.files(dir_path, pattern = pattern, full.names = TRUE)
  if (length(files) == 0L) return(NULL)
  # ファイル名でソートして最後を使う（YYYYMMDD付き前提）
  files[order(files)][length(files)]
}

# NbClust の結果から「best_k」と「全 k の index 値」のテーブルを安全に作る
build_nbclust_summary <- function(nb_res, indices, min_nc, max_nc) {
  # 返すオブジェクトの雛形
  best_df  <- data.frame(
    index        = indices,
    best_k       = NA_integer_,
    value_best_k = NA_real_,
    stringsAsFactors = FALSE
  )
  all_long <- data.frame(
    index = character(0),
    k     = integer(0),
    value = numeric(0),
    stringsAsFactors = FALSE
  )
  
  if (is.null(nb_res)) {
    return(list(best = best_df, all = all_long))
  }
  
  ## --- Best.nc から best_k, value_best_k を取得 ---
  if (!is.null(nb_res$Best.nc)) {
    best_nc <- nb_res$Best.nc
    # 行・列の状態に応じて処理
    if (is.matrix(best_nc) || is.data.frame(best_nc)) {
      # 期待する形: 行 = c("Number_clusters", "Value_Index"), 列 = インデックス名
      for (idx in indices) {
        if (idx %in% colnames(best_nc)) {
          bk  <- suppressWarnings(as.integer(best_nc["Number_clusters", idx]))
          val <- suppressWarnings(as.numeric(best_nc["Value_Index", idx]))
          best_df$best_k[best_df$index == idx]       <- bk
          best_df$value_best_k[best_df$index == idx] <- val
        }
      }
    } else if (is.vector(best_nc)) {
      # まれにベクトル形式で返るケース用の保険
      # 例: names: "Number_clusters.silhouette", "Value_Index.silhouette", ...
      for (idx in indices) {
        name_k   <- paste0("Number_clusters.", idx)
        name_val <- paste0("Value_Index.", idx)
        if (name_k %in% names(best_nc)) {
          bk <- suppressWarnings(as.integer(best_nc[[name_k]]))
          best_df$best_k[best_df$index == idx] <- bk
        }
        if (name_val %in% names(best_nc)) {
          val <- suppressWarnings(as.numeric(best_nc[[name_val]]))
          best_df$value_best_k[best_df$index == idx] <- val
        }
      }
    }
  }
  
  ## --- All.index から k vs value のロングテーブルを作る ---
  if (!is.null(nb_res$All.index)) {
    ai <- nb_res$All.index
    if (is.matrix(ai) || is.data.frame(ai)) {
      ai_df <- as.data.frame(ai)
      
      # 行名が k になっている想定だが、なければ min_nc:max_nc を使う
      if (!is.null(rownames(ai_df))) {
        ks <- suppressWarnings(as.integer(rownames(ai_df)))
        if (any(is.na(ks))) ks <- seq(min_nc, length.out = nrow(ai_df))
      } else {
        ks <- seq(min_nc, length.out = nrow(ai_df))
      }
      
      # 列インデックスごとにロング化
      for (idx in intersect(indices, colnames(ai_df))) {
        vals <- ai_df[[idx]]
        tmp <- data.frame(
          index = idx,
          k     = ks,
          value = as.numeric(vals),
          stringsAsFactors = FALSE
        )
        all_long <- rbind(all_long, tmp)
      }
    }
  }
  
  list(best = best_df, all = all_long)
}


## ---------------------------------------------------------------
## 2. Main loop over datasets (A–D)
## ---------------------------------------------------------------

cat("=== STEP2-variable: Variable-wise clustering on MDS (isoMDS) ===\n")
cat("ROOT =", ROOT, "\n")
cat("MDS dim for clustering =", mds_dim_for_clustering, "\n\n")

for (ds in datasets) {
  ds_id    <- ds$id
  subdir   <- ds$subdir
  
  cat("------------------------------------------------------------\n")
  cat("[INFO] Dataset", ds_id, "(", subdir, ")\n")
  
  input_dir  <- file.path(INPUT_BASE,  subdir)
  output_dir <- file.path(OUTPUT_BASE, subdir)
  
  # 出力フォルダ作成
  if (!dir.exists(output_dir)) {
    dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
    cat("[INFO] Created output directory:", output_dir, "\n")
  }
  
  # isoMDS 変数版の RDS ファイルを探す
  pattern_iso <- paste0("^mds_isoMDS_var_", ds_id, "_\\d{8}\\.rds$")
  iso_file <- find_latest_file(input_dir, pattern_iso)
  
  if (is.null(iso_file)) {
    cat("[WARN] No isoMDS variable file found in:", input_dir, "\n")
    cat("       Pattern:", pattern_iso, "\n")
    next
  }
  
  cat("[INFO] Using isoMDS variable file:", iso_file, "\n")
  
  # isoMDS オブジェクト読み込み
  mds_obj <- readRDS(iso_file)
  
  # isoMDS の座標を取得（MASS::isoMDS の返り値を想定）
  if (!("points" %in% names(mds_obj))) {
    stop("[ERROR] Loaded isoMDS object does not contain $points: ", iso_file)
  }
  
  coords_all <- as.matrix(mds_obj$points)
  n_var  <- nrow(coords_all)
  n_dim  <- ncol(coords_all)
  
  cat("[INFO] Number of variables:", n_var, "\n")
  cat("[INFO] Original MDS dimensions:", n_dim, "\n")
  
  use_dim <- min(mds_dim_for_clustering, n_dim)
  coords  <- coords_all[, seq_len(use_dim), drop = FALSE]
  
  # 変数名を取得（行名がなければ自動付与）
  var_names <- rownames(coords)
  if (is.null(var_names)) {
    var_names <- paste0("Var", seq_len(n_var))
  }
  
  ## -----------------------------------------------------------
  ## 2-1. Hierarchical clustering on MDS coordinates (Ward.D2)
  ## -----------------------------------------------------------
  
  cat("[INFO] Computing Euclidean distance on MDS space...\n")
  d_euclid <- dist(coords, method = "euclidean")
  
  cat("[INFO] Running hclust (method = 'ward.D2')...\n")
  hc <- hclust(d_euclid, method = "ward.D2")
  
  # hclust オブジェクトを保存（任意だが将来用に）
  hclust_outfile <- file.path(
    output_dir,
    sprintf("hclust_var_wardD2_%s_%s.rds", ds_id, run_date)
  )
  saveRDS(hc, hclust_outfile)
  cat("[SAVED] hclust object:", hclust_outfile, "\n")
  
  ## -----------------------------------------------------------
  ## 2-2. クラスタラベル（k = 2〜25）を wide 形式で保存
  ## -----------------------------------------------------------
  
  cat("[INFO] Cutting tree into clusters for k =", min_nc, "to", max_nc, "...\n")
  
  label_mat <- sapply(min_nc:max_nc, function(k) {
    cutree(hc, k = k)
  })
  colnames(label_mat) <- paste0("cluster_k", min_nc:max_nc)
  rownames(label_mat) <- var_names
  
  labels_df <- data.frame(
    variable_name = var_names,
    label_mat,
    check.names = FALSE,
    row.names = NULL
  )
  
  labels_outfile <- file.path(
    output_dir,
    sprintf("nbclust_var_labels_%s_%s.csv", ds_id, run_date)
  )
  write.csv(labels_df, labels_outfile, row.names = FALSE)
  cat("[SAVED] Variable cluster labels (wide):", labels_outfile, "\n")
  
  ## -----------------------------------------------------------
  ## 2-3. NbClust を用いたクラスタ数推奨 & 指標値のサマリ
  ## -----------------------------------------------------------
  
  cat("[INFO] Running NbClust (indices:", paste(nb_indices, collapse = ", "),
      "; k from", min_nc, "to", max_nc, ") ...\n")
  
  nb_res <- NULL
  nb_error <- NULL
  
  nb_res <- tryCatch(
    {
      NbClust(
        data    = coords,
        distance = "euclidean",
        min.nc   = min_nc,
        max.nc   = max_nc,
        method   = "ward.D2",
        index    = nb_indices
      )
    },
    error = function(e) {
      nb_error <<- e
      NULL
    }
  )
  
  if (!is.null(nb_error)) {
    cat("[WARN] NbClust failed for dataset", ds_id, ":\n")
    cat("       ", conditionMessage(nb_error), "\n")
  } else {
    cat("[INFO] NbClust finished successfully for dataset", ds_id, "\n")
  }
  
  # NbClust 結果からサマリテーブルを作成
  summary_list <- build_nbclust_summary(nb_res, nb_indices, min_nc, max_nc)
  best_df  <- summary_list$best
  all_long <- summary_list$all
  
  # best_k を all_long にマージして、1ファイルにまとめる
  if (nrow(all_long) > 0) {
    nb_summary <- merge(
      all_long,
      best_df,
      by = "index",
      all.x = TRUE,
      sort = FALSE
    )
    nb_summary$is_best <- with(nb_summary, ifelse(!is.na(best_k) & k == best_k, TRUE, FALSE))
  } else {
    # All.index が無い場合は best_df のみ
    nb_summary <- best_df
    nb_summary$k     <- NA_integer_
    nb_summary$value <- NA_real_
    nb_summary$is_best <- NA
    # 列順を合わせる
    nb_summary <- nb_summary[, c("index", "k", "value", "best_k", "value_best_k", "is_best")]
  }
  
  # 列の順序を明示
  if (all(c("index", "k", "value", "best_k", "value_best_k", "is_best") %in% names(nb_summary))) {
    nb_summary <- nb_summary[, c("index", "k", "value", "best_k", "value_best_k", "is_best")]
  }
  
  summary_outfile <- file.path(
    output_dir,
    sprintf("nbclust_var_summary_%s_%s.csv", ds_id, run_date)
  )
  write.csv(nb_summary, summary_outfile, row.names = FALSE)
  cat("[SAVED] NbClust summary (variable-wise):", summary_outfile, "\n")
  
  cat("[INFO] Finished dataset", ds_id, "\n\n")
}

cat("=== STEP2-variable completed. ===\n")


=== STEP2-variable: Variable-wise clustering on MDS (isoMDS) ===
ROOT = ~/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127 
MDS dim for clustering = 10 

------------------------------------------------------------
[INFO] Dataset A ( A_OH_plus_others )
[WARN] No isoMDS variable file found in: ~/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core_var/01_distance_mds_variables/A_OH_plus_others 
       Pattern: ^mds_isoMDS_var_A_\d{8}\.rds$ 
------------------------------------------------------------
[INFO] Dataset B ( B_FP_plus_others )
[WARN] No isoMDS variable file found in: ~/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/data/calc_core_var/01_distance_mds_variables/B_FP_plus_others 
       Pattern: ^mds_isoMDS_var_B_\d{8}\.rds$ 
------------------------------------------------------------
[INFO] Dataset C ( C_OH_only )
[WARN] No isoMDS variable file found in: ~/cal_cluste